# 05 -- BESS / VPP / Solar Comparator

**Purpose (PROJECT.md Section 8.7):** extend the fabric-retrofit model with a BESS (battery energy storage system) + rooftop solar + VPP fit-out, modelled as a comparator and complement to fabric retrofit -- not a replacement for it. Week 5.

**Why this notebook exists:** notebooks 01-04 answered "how much flexible grid capacity does insulation create?" and found a real but limited, currently non-contractible answer (Section 3, `notebooks/04`). The natural next question: how does that compare to an asset that genuinely *can* be metered, dispatched, and paid for today? This notebook builds that comparison using the same archetype, the same peak-window event, and the same tenure-fragmentation lens already established -- not a separate project.

**Scope boundary (PROJECT.md Section 2.2, Sledgehammer Test):** this reuses researched real-world UK 2026 market figures for solar/battery cost, generation, and VPP/arbitrage revenue (`configs/tenure_insulation_assumptions.yml`, `bess_solar_vpp`) rather than building a new solar-irradiance or annual battery-dispatch simulation from scratch. It is a physics-and-market comparator, not a merchant dispatch model -- consistent with how `notebooks/04` treats fabric economics.

**Three comparisons, per the project's scoping decision:**
1. **Fabric-only** (notebooks 01-04's existing result)
2. **BESS+solar-only** (no fabric retrofit)
3. **Fabric + BESS stacked** (does retrofit change what size/cost of battery is needed for the same outcome?)

Plus a tenure-adoption comparison, since this project's core thesis is about who controls retrofit decisions -- and BESS/solar turns out to have a different, and in one respect sharper, version of that same problem.

In [1]:
import sys

print("Python executable:", sys.executable)
assert "thermal-counterfactual-gb" in sys.executable, (
    "Wrong kernel selected -- pick the 'thermal-counterfactual-gb' kernel, "
    "not a default/global one. Run setup.sh first if it doesn't exist yet."
)

import numpy as np
import polars as pl
import yaml
print("polars:", pl.__version__, "| numpy:", np.__version__)

Python executable: /tmp/kernelenv/thermal-counterfactual-gb/bin/python3
polars: 1.43.2 | numpy: 2.2.6


## 1. Load assumptions and Notebook 01/03/04 outputs

Notebook 01's resolved H/tau/coastdown/peak-kW figures live directly in config (`resolved_physics`) once hand-verified there, per the Traceability Mandate -- no need to re-read a notebook-01 parquet just for those three numbers. Notebook 03's tenure-weighted fabric prevalence and Notebook 04's delivered-battery-equivalent figure are read from their parquet handoffs, per the Parquet Handoff Rule (PROJECT.md Section 4.2).

In [2]:
from pathlib import Path

CONFIG_PATH = Path("../configs/tenure_insulation_assumptions.yml")
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

population_03 = pl.read_parquet("../data/intermediate/03_estate_population_model.parquet")
economics_04 = pl.read_parquet("../data/intermediate/04_vpp_economics.parquet")

resolved = cfg["resolved_physics"]
bess = cfg["bess_solar_vpp"]

peak_window_str = cfg["cold_snap_event"]["peak_window"]
start_str, end_str = peak_window_str.split("-")
peak_hours = int(end_str.split(":")[0]) - int(start_str.split(":")[0])

fabric_prevalence_weighted = population_03["weighted_insulation_prevalence"][0]
delivered_kwh_event_epc_c = economics_04["delivered_kwh_event_epc_c"][0]

print(f"Peak window: {peak_window_str} ({peak_hours} hours)")
print(f"Fabric retrofit weighted estate prevalence (Notebook 03): {fabric_prevalence_weighted:.1%}")
print(f"EPC-C fabric delivered flexibility in the real Dec 2022 event (Notebook 04): {delivered_kwh_event_epc_c:.1f} kWh_e/home")

Peak window: 16:00-20:00 (4 hours)
Fabric retrofit weighted estate prevalence (Notebook 03): 27.0%
EPC-C fabric delivered flexibility in the real Dec 2022 event (Notebook 04): 2.2 kWh_e/home


## 2. Can a battery physically cover the peak-window demand? (power rating, not just energy)

Two different things are easy to conflate: a battery's *power rating* (kW -- how fast it can discharge right now) and its *energy capacity* (kWh -- how long it can sustain that). Fabric retrofit changes a home's continuous peak-window electrical demand (`resolved_physics`, Section 8.3); a battery's power rating determines whether it can physically supply that demand at all, independent of how long it lasts.

In [3]:
power_rating = bess["battery_hardware"]["continuous_power_rating_kw"]
states = ["baseline", "swi_only", "epc_c"]
labels = {"baseline": "Baseline (unretrofitted)", "swi_only": "SWI only", "epc_c": "EPC-C package"}

print(f"Domestic battery continuous power rating: {power_rating['low']}-{power_rating['high']} kW (point estimate {power_rating['point']} kW)")
print("Source: Tesla Powerwall 3 / GivEnergy All-in-One 2 product data, 2026 (GROUNDED).")
print()
for s in states:
    peak_kw = resolved[s]["peak_kw_at_cop2_5"]
    covers_at_low_end = power_rating["low"] >= peak_kw
    print(f"{labels[s]:24s}: peak demand {peak_kw:.2f} kW -- battery covers it even at the LOW end of the "
          f"power-rating range: {covers_at_low_end}")

print()
print(
    "A domestic battery's power rating is never the binding constraint here, even for the least-capable "
    "product on the market and the least-retrofitted fabric state. Retrofit does not change whether a "
    "battery CAN cover peak demand -- it changes for how LONG a given battery can do so, which is an "
    "energy (kWh), not power (kW), question. That's Section 3 below."
)

Domestic battery continuous power rating: 3.6-11.5 kW (point estimate 5.0 kW)
Source: Tesla Powerwall 3 / GivEnergy All-in-One 2 product data, 2026 (GROUNDED).

Baseline (unretrofitted): peak demand 2.46 kW -- battery covers it even at the LOW end of the power-rating range: True
SWI only                : peak demand 2.06 kW -- battery covers it even at the LOW end of the power-rating range: True
EPC-C package           : peak demand 0.66 kW -- battery covers it even at the LOW end of the power-rating range: True

A domestic battery's power rating is never the binding constraint here, even for the least-capable product on the market and the least-retrofitted fabric state. Retrofit does not change whether a battery CAN cover peak demand -- it changes for how LONG a given battery can do so, which is an energy (kWh), not power (kW), question. That's Section 3 below.


## 3. Energy needed for full peak coverage -- the real constraint, and the fabric+battery synergy

Assume a battery alone (no solar contribution -- see Section 4 for why) is asked to fully cover a home's peak-window demand, continuously, for the full 4-hour window: energy needed = peak kW x peak hours. Compare against a standard battery's usable capacity (10 kWh, `bess_solar_vpp.battery_hardware`).

In [4]:
usable_capacity_kwh = bess["battery_hardware"]["usable_capacity_kwh"]

print(f"Standard battery usable capacity: {usable_capacity_kwh} kWh")
print()
energy_needed = {}
for s in states:
    peak_kw = resolved[s]["peak_kw_at_cop2_5"]
    kwh_needed = peak_kw * peak_hours
    energy_needed[s] = kwh_needed
    utilisation = kwh_needed / usable_capacity_kwh
    fits = kwh_needed <= usable_capacity_kwh
    print(f"{labels[s]:24s}: {peak_kw:.2f} kW x {peak_hours}h = {kwh_needed:.2f} kWh needed for FULL peak "
          f"coverage -- {utilisation:.0%} of a standard battery's usable capacity (fits with room to spare: {fits})")

print()
print(
    f"Cross-check against Notebook 04: EPC-C fabric's own coastdown ALREADY delivered {delivered_kwh_event_epc_c:.1f} "
    f"kWh_e/home in the real (milder-than-design) Dec 2022 event -- close to this section's {energy_needed['epc_c']:.2f} "
    f"kWh design-case figure, as expected since they describe the same physical mechanism from two directions."
)
print()
print(
    "The synergy: pairing fabric retrofit with a battery does not just add two independent benefits -- it "
    "changes how HARD the battery has to work for the same outcome (zero grid draw during the peak window). "
    "Unretrofitted fabric asks a standard 10 kWh battery to run right at the edge of its usable capacity every "
    "single evening; EPC-C fabric leaves the same battery two-thirds empty after doing the same job -- headroom "
    "that buys resilience against multi-day cold snaps, battery degradation over its lifetime, or simply not "
    "needing to fully cycle the battery every night."
)

Standard battery usable capacity: 10 kWh

Baseline (unretrofitted): 2.46 kW x 4h = 9.84 kWh needed for FULL peak coverage -- 98% of a standard battery's usable capacity (fits with room to spare: True)
SWI only                : 2.06 kW x 4h = 8.24 kWh needed for FULL peak coverage -- 82% of a standard battery's usable capacity (fits with room to spare: True)
EPC-C package           : 0.66 kW x 4h = 2.64 kWh needed for FULL peak coverage -- 26% of a standard battery's usable capacity (fits with room to spare: True)

Cross-check against Notebook 04: EPC-C fabric's own coastdown ALREADY delivered 2.2 kWh_e/home in the real (milder-than-design) Dec 2022 event -- close to this section's 2.64 kWh design-case figure, as expected since they describe the same physical mechanism from two directions.

The synergy: pairing fabric retrofit with a battery does not just add two independent benefits -- it changes how HARD the battery has to work for the same outcome (zero grid draw during the peak wind

## 4. Solar's role: annual value, not winter-evening peak-shaving

Important scope clarification, easy to get wrong: the peak window this whole project cares about is 16:00-20:00 in a **December** cold snap -- after sunset, or close to it, in North West England. Solar PV generates essentially nothing during that specific window. Solar's economic value here is an annual, mostly-daytime/summer figure (self-consumption and export across the whole year), not a contributor to the winter-evening peak-shaving mechanism modelled in Sections 2-3. Conflating the two would be a real error, not just an imprecision -- flagging it explicitly rather than letting a reader assume solar helps with THIS peak.

In [5]:
solar = bess["solar_pv"]
system_kwp = solar["system_size_kwp"]
yield_point = solar["specific_yield_kwh_per_kwp_per_year"]["point"]
derate = solar["orientation_derate_factor"]["value"]
annual_generation = system_kwp * yield_point * derate

print(f"Representative system: {system_kwp} kWp, {yield_point} kWh/kWp/yr UK average, "
      f"x {derate} orientation derate (non-ideal terrace roof orientation, DELIBERATE)")
print(f"Estimated annual generation: {annual_generation:.0f} kWh/year (PROVISIONAL, context figure only)")
print()
print(
    "This generation is real and valuable across the year, but it is NOT what makes a battery able to cover "
    "the December 16:00-20:00 peak window in Sections 2-3 -- that's entirely the battery's own stored energy, "
    "charged overnight off-peak (the arbitrage mechanism in Section 6) or from summer/daytime solar surplus "
    "stored earlier. Solar's contribution to THIS project's specific winter-evening flexibility question is "
    "close to zero; its contribution to the battery owner's ANNUAL economics (Section 6) is real and separate. "
    "Roof-area and conservation-area constraints on achieving even this 4 kWp figure are noted in config "
    "(bess_solar_vpp.solar_pv.roof_constraint_note) but not independently checked for this archetype."
)

Representative system: 4.0 kWp, 950 kWh/kWp/yr UK average, x 0.85 orientation derate (non-ideal terrace roof orientation, DELIBERATE)
Estimated annual generation: 3230 kWh/year (PROVISIONAL, context figure only)

This generation is real and valuable across the year, but it is NOT what makes a battery able to cover the December 16:00-20:00 peak window in Sections 2-3 -- that's entirely the battery's own stored energy, charged overnight off-peak (the arbitrage mechanism in Section 6) or from summer/daytime solar surplus stored earlier. Solar's contribution to THIS project's specific winter-evening flexibility question is close to zero; its contribution to the battery owner's ANNUAL economics (Section 6) is real and separate. Roof-area and conservation-area constraints on achieving even this 4 kWp figure are noted in config (bess_solar_vpp.solar_pv.roof_constraint_note) but not independently checked for this archetype.


## 5. Cost of a "right-sized" battery: what retrofit saves on hardware

Using the energy-needed figures from Section 3 and the GROUNDED battery installed-cost range (`vpp_economics.domestic_battery_installed_cost_gbp_per_kwh`), a purely illustrative marginal-hardware-cost comparison: what would a battery sized ONLY for each fabric state's full-coverage requirement cost, at a flat GBP/kWh rate?

**DELIBERATE simplification, stated plainly:** real batteries are sold in discrete standard sizes (5 kWh, 9.5 kWh, 10 kWh, 13.5 kWh...), not continuously, and this ignores fixed inverter/installation costs that don't scale with kWh. This is a theoretical marginal-energy comparison to show the DIRECTION and rough SCALE of the synergy, not a literal "you could buy a 3.28 kWh battery" product claim.

In [6]:
battery_cost_per_kwh_point = cfg["vpp_economics"]["domestic_battery_installed_cost_gbp_per_kwh"]["point"]

print(f"Illustrative flat rate: GBP {battery_cost_per_kwh_point}/kWh installed (GROUNDED range, point estimate)")
print()
marginal_cost = {}
for s in states:
    cost = energy_needed[s] * battery_cost_per_kwh_point
    marginal_cost[s] = cost
    print(f"{labels[s]:24s}: {energy_needed[s]:.2f} kWh needed -> GBP {cost:,.0f} illustrative marginal battery hardware cost")

saving = marginal_cost["baseline"] - marginal_cost["epc_c"]
print()
print(f"Illustrative hardware-cost saving from pairing retrofit with a right-sized battery, "
      f"baseline vs EPC-C: GBP {saving:,.0f} -- roughly the same ~68% reduction as the peak-kW figure itself, "
      "since this is a simple linear scaling of it. Real-world savings would be smaller and lumpier, since "
      "both cases would likely buy the same smallest standard product on the market rather than a bespoke size.")

Illustrative flat rate: GBP 650/kWh installed (GROUNDED range, point estimate)

Baseline (unretrofitted): 9.84 kWh needed -> GBP 6,396 illustrative marginal battery hardware cost
SWI only                : 8.24 kWh needed -> GBP 5,356 illustrative marginal battery hardware cost
EPC-C package           : 2.64 kWh needed -> GBP 1,716 illustrative marginal battery hardware cost

Illustrative hardware-cost saving from pairing retrofit with a right-sized battery, baseline vs EPC-C: GBP 4,680 -- roughly the same ~68% reduction as the peak-kW figure itself, since this is a simple linear scaling of it. Real-world savings would be smaller and lumpier, since both cases would likely buy the same smallest standard product on the market rather than a bespoke size.


## 6. Three-way economics: fabric-only vs BESS+solar-only vs stacked

The central asymmetry from `notebooks/04`, carried forward: fabric's flexibility value is illustrative-if-contractible (category error, not real revenue); BESS/VPP's is real and currently contractible. Both cost figures are GROUNDED; both revenue/value figures are shown honestly labelled by kind, not blended into one misleading "ROI" number.

In [7]:
fabric_cost = cfg["retrofit_cost_gbp_to_epc_c"]["pre_1919_age_band_average"]
fabric_illustrative_value = economics_04["dno_flex_value_gbp_per_home_per_year_point"][0]
fabric_delivered_battery_equiv = economics_04["delivered_value_gbp_per_home_point"][0]

bess_cost = bess["installed_cost_gbp_bundled_4kwp_10kwh"]["point"]
bess_annual_revenue = bess["vpp_and_arbitrage_earnings_gbp_per_year"]["combined_point"]
bess_payback_years = bess_cost / bess_annual_revenue

stacked_cost = fabric_cost + marginal_cost["epc_c"]

print("FABRIC-ONLY (Notebooks 01-04):")
print(f"  Cost: GBP {fabric_cost:,.0f}/home (GROUNDED, EHS age-band average)")
print(f"  Annual flexibility value IF contractible: GBP {fabric_illustrative_value:.0f}/home/year -- NOT real revenue (category error, Notebook 04 Section 3)")
print(f"  One-off delivered battery-equivalent value (real Dec 2022 event): GBP {fabric_delivered_battery_equiv:,.0f}/home")
print()
print("BESS+SOLAR-ONLY (no fabric retrofit):")
print(f"  Cost: GBP {bess_cost:,.0f}/home (GROUNDED, bundled 4kWp+10kWh 2026 UK market survey)")
print(f"  Annual revenue: GBP {bess_annual_revenue:.0f}/home/year -- REAL and currently contractible (VPP dispatch + arbitrage, GROUNDED)")
print(f"  Simple payback at that real revenue: {bess_payback_years:.1f} years")
print(f"  Covers baseline's full peak-window demand at ~{energy_needed['baseline']/usable_capacity_kwh:.0%} of usable capacity (Section 3) -- tight but sufficient")
print()
print("FABRIC + BESS STACKED (illustrative, marginal-cost battery per Section 5):")
print(f"  Cost: GBP {stacked_cost:,.0f}/home (fabric GBP {fabric_cost:,.0f} + right-sized battery GBP {marginal_cost['epc_c']:,.0f})")
print(f"  Higher upfront cost than BESS alone, but: comfort/health/EPC benefits fabric alone provides, "
      f"PLUS a battery running at only ~{energy_needed['epc_c']/usable_capacity_kwh:.0%} of usable capacity for the "
      f"same peak-coverage outcome -- meaning a smaller real product could plausibly be chosen, or the same "
      f"product retains headroom for multi-day events and degradation.")
print()
print(
    "The honest bottom line: BESS+solar-only is the only one of the three with REAL, currently-contractible "
    "annual revenue. Fabric-only has no real revenue mechanism today but delivers comfort, health and EPC "
    "compliance that a battery alone does not touch. They are complements answering different questions, "
    "not competing investments with a single comparable payback -- collapsing them into one number would "
    "repeat the same category error Notebook 04 already corrected once."
)

FABRIC-ONLY (Notebooks 01-04):
  Cost: GBP 10,728/home (GROUNDED, EHS age-band average)
  Annual flexibility value IF contractible: GBP 127/home/year -- NOT real revenue (category error, Notebook 04 Section 3)
  One-off delivered battery-equivalent value (real Dec 2022 event): GBP 1,447/home

BESS+SOLAR-ONLY (no fabric retrofit):
  Cost: GBP 13,500/home (GROUNDED, bundled 4kWp+10kWh 2026 UK market survey)
  Annual revenue: GBP 1075/home/year -- REAL and currently contractible (VPP dispatch + arbitrage, GROUNDED)
  Simple payback at that real revenue: 12.6 years
  Covers baseline's full peak-window demand at ~98% of usable capacity (Section 3) -- tight but sufficient

FABRIC + BESS STACKED (illustrative, marginal-cost battery per Section 5):
  Cost: GBP 12,444/home (fabric GBP 10,728 + right-sized battery GBP 1,716)
  Higher upfront cost than BESS alone, but: comfort/health/EPC benefits fabric alone provides, PLUS a battery running at only ~26% of usable capacity for the same peak-cover

## 7. Tenure-weighted adoption: BESS/solar today vs fabric, and the split-incentive twist

This project's core thesis is about who controls retrofit decisions across a fragmented estate. BESS/solar turns out to have a strikingly different adoption pattern from fabric -- worth reporting as a real, quantified finding, not an assumption.

In [8]:
adopt = bess["adoption_by_tenure"]
tenure_mix = cfg["tenure_mix"]

tenure_labels = {
    "local_authority_retained": "Local authority (retained)",
    "housing_association_retained": "Housing association (retained)",
    "ex_rtb_privately_rented": "Ex-RTB, privately rented",
    "ex_rtb_owner_occupied": "Ex-RTB, owner-occupied",
}
adopt_key_map = {
    "local_authority_retained": "local_authority",
    "housing_association_retained": "housing_association",
    "ex_rtb_privately_rented": "ex_rtb_privately_rented",
    "ex_rtb_owner_occupied": "ex_rtb_owner_occupied",
}

print("BESS/solar adoption rate by tenure (PROVISIONAL, derived -- see config derivation_note):")
weighted = 0.0
for t, label in tenure_labels.items():
    rate = adopt[adopt_key_map[t]]
    share = tenure_mix[t]
    weighted += rate * share
    print(f"  {label:32s}: {rate:.1%} of homes in this segment (estate share {share:.0%})")

print()
print(f"Estate-weighted BESS/solar adoption today: {weighted:.1%}")
print(f"Estate-weighted FABRIC retrofit adoption today (Notebook 03): {fabric_prevalence_weighted:.1%}")
print(f"Ratio: fabric adoption is {fabric_prevalence_weighted/weighted:.1f}x higher than BESS/solar adoption on this same estate")
print()
print(
    "This is a real inversion worth sitting with: the asset with NO working revenue mechanism today (fabric, "
    "Notebook 04 Section 3's category error) has been deployed roughly 4-5x more than the asset WITH a real, "
    "currently-contractible revenue mechanism (BESS/VPP). The reason is not physics or economics -- it's "
    "delivery mechanism. Fabric retrofit has been landlord-funded and programme-driven (Warm Homes Social "
    "Housing Fund, ring-fenced HRA capital); solar+battery has historically been an individual owner-occupier "
    "consumer purchase, not a landlord rollout."
)
print()
print(bess["adoption_by_tenure"]["policy_context"].strip())
print()
print("Split-incentive note (config: bess_solar_vpp.split_incentive_note):")
print(bess["split_incentive_note"]["note"].strip())

BESS/solar adoption rate by tenure (PROVISIONAL, derived -- see config derivation_note):
  Local authority (retained)      : 5.2% of homes in this segment (estate share 24%)
  Housing association (retained)  : 5.2% of homes in this segment (estate share 41%)
  Ex-RTB, privately rented        : 5.2% of homes in this segment (estate share 15%)
  Ex-RTB, owner-occupied          : 8.3% of homes in this segment (estate share 20%)

Estate-weighted BESS/solar adoption today: 5.8%
Estate-weighted FABRIC retrofit adoption today (Notebook 03): 27.0%
Ratio: fabric adoption is 4.6x higher than BESS/solar adoption on this same estate

This is a real inversion worth sitting with: the asset with NO working revenue mechanism today (fabric, Notebook 04 Section 3's category error) has been deployed roughly 4-5x more than the asset WITH a real, currently-contractible revenue mechanism (BESS/VPP). The reason is not physics or economics -- it's delivery mechanism. Fabric retrofit has been landlord-funded a

## 8. Modelling Prose check (PROJECT.md Section 8.3)

In [9]:
headline = (
    f"A domestic battery's POWER rating is never the constraint on covering this project's winter evening "
    f"peak window, at any fabric state -- but its ENERGY capacity is: an unretrofitted home asks a standard "
    f"10 kWh battery to run at {energy_needed['baseline']/usable_capacity_kwh:.0%} of usable capacity every evening to fully cover its own "
    f"peak demand, while EPC-C fabric leaves the same battery at only {energy_needed['epc_c']/usable_capacity_kwh:.0%} utilisation for the same "
    f"outcome. Unlike fabric retrofit's illustrative-if-contractible flexibility value (Notebook 04), a "
    f"BESS+solar fit-out's GBP {bess_annual_revenue:.0f}/home/year VPP-and-arbitrage revenue is real and "
    f"currently contractible today, giving a genuine {bess_payback_years:.1f}-year simple payback on its "
    f"GBP {bess_cost:,.0f}/home installed cost -- a materially different, and more real, economic case than "
    f"fabric's. Yet on this same estate, fabric retrofit adoption ({fabric_prevalence_weighted:.1%}) is roughly "
    f"{fabric_prevalence_weighted/weighted:.1f}x higher than BESS/solar adoption ({weighted:.1%}) today, because "
    f"fabric has been landlord-funded and programme-driven while solar+battery has historically been an "
    f"individual purchase -- and BESS/VPP's ongoing revenue typically accrues to the bill-paying tenant, not "
    f"the landlord who would fund the install, a sharper split-incentive problem than fabric's own."
)
print(headline)
print()
print("Modelling Prose check (PROJECT.md Section 8.3 pattern):")
print(f"  Number/Unit:    {bess_annual_revenue:.0f} GBP/home/year real BESS revenue; {bess_payback_years:.1f}-year payback; "
      f"{fabric_prevalence_weighted:.1%} fabric vs {weighted:.1%} BESS/solar estate-weighted adoption")
print("  Denominator:    per home; per year for revenue and adoption-rate figures; one-off for install costs")
print("  Mechanism:      a battery is metered/dispatchable by construction so it can bid into VPP/arbitrage markets fabric cannot; retrofit does not change whether a battery CAN cover peak demand, only how much of its capacity that costs")
print("  Scope boundary: no hour-by-hour solar generation or dispatch simulation; battery degradation, round-trip losses, and inverter clipping not modelled; marginal battery-cost figures are illustrative, not real product-SKU pricing")
print("  Caveat:         BESS/solar adoption-by-tenure figures are PROVISIONAL (derived from share-of-owners data, not a directly published rate); solar contributes ~nothing to the specific Dec evening peak window modelled; the three options are complements answering different questions, not one blended ROI")

A domestic battery's POWER rating is never the constraint on covering this project's winter evening peak window, at any fabric state -- but its ENERGY capacity is: an unretrofitted home asks a standard 10 kWh battery to run at 98% of usable capacity every evening to fully cover its own peak demand, while EPC-C fabric leaves the same battery at only 26% utilisation for the same outcome. Unlike fabric retrofit's illustrative-if-contractible flexibility value (Notebook 04), a BESS+solar fit-out's GBP 1075/home/year VPP-and-arbitrage revenue is real and currently contractible today, giving a genuine 12.6-year simple payback on its GBP 13,500/home installed cost -- a materially different, and more real, economic case than fabric's. Yet on this same estate, fabric retrofit adoption (27.0%) is roughly 4.6x higher than BESS/solar adoption (5.8%) today, because fabric has been landlord-funded and programme-driven while solar+battery has historically been an individual purchase -- and BESS/VPP's

## 9. Save to `data/intermediate/`

In [10]:
summary = pl.DataFrame([{
    "battery_power_rating_kw_point": power_rating["point"],
    "energy_needed_kwh_baseline": energy_needed["baseline"],
    "energy_needed_kwh_swi_only": energy_needed["swi_only"],
    "energy_needed_kwh_epc_c": energy_needed["epc_c"],
    "battery_usable_capacity_kwh": usable_capacity_kwh,
    "solar_estimated_annual_generation_kwh": annual_generation,
    "fabric_cost_gbp_per_home": fabric_cost,
    "bess_cost_gbp_per_home": bess_cost,
    "stacked_cost_gbp_per_home": stacked_cost,
    "bess_annual_revenue_gbp_per_home": bess_annual_revenue,
    "bess_payback_years": bess_payback_years,
    "fabric_weighted_adoption": fabric_prevalence_weighted,
    "bess_weighted_adoption": weighted,
    "adoption_ratio_fabric_to_bess": fabric_prevalence_weighted / weighted,
}])

out_path = Path("../data/intermediate/05_bess_vpp_solar_comparator.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)
summary.write_parquet(out_path)
print(f"Wrote {out_path} ({summary.height} rows)")
summary

Wrote ../data/intermediate/05_bess_vpp_solar_comparator.parquet (1 rows)


battery_power_rating_kw_point,energy_needed_kwh_baseline,energy_needed_kwh_swi_only,energy_needed_kwh_epc_c,battery_usable_capacity_kwh,solar_estimated_annual_generation_kwh,fabric_cost_gbp_per_home,bess_cost_gbp_per_home,stacked_cost_gbp_per_home,bess_annual_revenue_gbp_per_home,bess_payback_years,fabric_weighted_adoption,bess_weighted_adoption,adoption_ratio_fabric_to_bess
f64,f64,f64,f64,i64,f64,i64,i64,f64,i64,f64,f64,f64,f64
5.0,9.84,8.24,2.64,10,3230.0,10728,13500,12444.0,1075,12.55814,0.2703,0.0582,4.64433
